# Bayesian Linear Regression

Companion notebook for the [Bayesian Linear Regression lesson](https://ml-viz-ruby.vercel.app/courses/bayesian-methods/01-bayesian-linear-regression).

**The idea in one sentence.** Instead of a single best-fit line, Bayesian linear
regression keeps a whole **distribution over lines** — a Gaussian posterior over
the weights — which gives you *calibrated error bars* that widen where you have no
data.

Two connections make it click:

- **The posterior mean is exactly ridge regression** (with $\lambda = \alpha/\beta$),
  so the Bayesian machinery *contains* the regularized point estimate you already know.
- **The predictive variance has two parts:** an irreducible noise floor $1/\beta$
  plus an *epistemic* term that grows as you extrapolate away from the data.

We derive the closed-form Gaussian posterior, **validate it against ridge and show
it collapses to OLS as the prior vanishes**, then cover the gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — Data and design matrix

A few noisy points from a linear function, clustered on the left so we can see what happens where
there is no data. Features are [1, x] (bias + slope).

In [ ]:
x = np.array([-2.0, -1.6, -1.2, -0.8, -0.4, 0.0])     # data only on the left
y = 0.9 * x + 0.3 + rng.normal(0, 0.15, size=x.shape)
Phi = np.c_[np.ones_like(x), x]                        # design matrix [1, x]
alpha, beta = 2.0, 25.0                                 # prior precision, noise precision

## 2 — The closed-form Gaussian posterior over weights

S_N^{-1} = αI + β ΦᵀΦ ;  m_N = β S_N Φᵀy. With Gaussian prior and likelihood the posterior is
Gaussian — no sampling needed.

In [ ]:
def posterior(Phi, y, alpha, beta):
    d = Phi.shape[1]
    S_N_inv = alpha * np.eye(d) + beta * Phi.T @ Phi
    S_N = np.linalg.inv(S_N_inv)
    m_N = beta * S_N @ Phi.T @ y
    return m_N, S_N

m_N, S_N = posterior(Phi, y, alpha, beta)
print('posterior mean weights [bias, slope]:', m_N.round(3))
print('posterior covariance:\n', S_N.round(4))

## 3 — The posterior mean IS ridge regression

Ridge with λ = α/β should reproduce the posterior mean exactly.

In [ ]:
lam = alpha / beta
ridge_w = np.linalg.solve(Phi.T @ Phi + lam * np.eye(2), Phi.T @ y)
print('ridge (λ=α/β) weights: ', ridge_w.round(3))
print('posterior mean weights:', m_N.round(3))
assert np.allclose(ridge_w, m_N)
print('\u2713 posterior mean equals the ridge solution')

### Validate: the prior strength interpolates ridge ↔ OLS

The posterior mean equals ridge with $\lambda=\alpha/\beta$ (asserted above). As
the prior precision $\alpha \to 0$ the penalty vanishes, so the posterior mean must
converge to the **ordinary least squares** solution — the bridge between Bayesian
and classical regression.

In [ ]:
ols = np.linalg.lstsq(Phi, y, rcond=None)[0]
m_weak, _ = posterior(Phi, y, alpha=1e-8, beta=beta)      # essentially no prior
m_strong, _ = posterior(Phi, y, alpha=100.0, beta=beta)   # strong prior -> shrink to 0
print(f'OLS weights            : {ols.round(3)}')
print(f'posterior (alpha->0)   : {m_weak.round(3)}  (matches OLS)')
print(f'posterior (alpha=100)  : {m_strong.round(3)}  (shrunk toward 0)')
assert np.allclose(m_weak, ols, atol=1e-4), 'vanishing prior => OLS'
assert np.linalg.norm(m_strong) < np.linalg.norm(ols), 'strong prior shrinks the weights'
print('\n✅ alpha tunes prior strength: ~0 recovers OLS, large shrinks toward the prior mean (0)')

## 4 — Predictive distribution: error bars widen away from data

σ²(x*) = β⁻¹ + φ(x*)ᵀ S_N φ(x*). We plot the predictive mean ± 2σ across a range. Notice the band
is tight over the data (left) and flares out where we extrapolate (right).

In [ ]:
xs = np.linspace(-3, 3, 200)
Phis = np.c_[np.ones_like(xs), xs]
mean = Phis @ m_N
var = 1.0 / beta + np.einsum('ij,jk,ik->i', Phis, S_N, Phis)
sd = np.sqrt(var)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(xs, mean - 2*sd, mean + 2*sd, color='#6366f1', alpha=0.3, label='±2σ predictive')
ax.plot(xs, mean, color='#818cf8', label='predictive mean')
ax.scatter(x, y, color='#2dd4bf', zorder=5, label='data')
ax.axvspan(x.min(), x.max(), color='#2dd4bf', alpha=0.05)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Predictive uncertainty is tight on the data, flares out in extrapolation')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'predictive sd at x=0 (in data):  {np.sqrt(1/beta + np.r_[1,0]@S_N@np.r_[1,0]):.3f}')
print(f'predictive sd at x=3 (far away):  {np.sqrt(1/beta + np.r_[1,3]@S_N@np.r_[1,3]):.3f}')

### Validate: predictive uncertainty splits into noise + epistemic

The predictive variance is $1/\beta + \phi_*^\top S_N \phi_*$. The first term is
the irreducible observation noise (a floor you can never beat); the second is
*epistemic* uncertainty that shrinks with data and grows in extrapolation. We
confirm the variance never dips below the noise floor and is larger far from the
data than inside it.

In [ ]:
def pred_var(phi_star):
    return 1.0 / beta + phi_star @ S_N @ phi_star

v_in  = pred_var(np.r_[1.0, 0.0])    # x=0, inside the data span
v_out = pred_var(np.r_[1.0, 3.0])    # x=3, far in extrapolation
print(f'noise floor 1/beta          : {1/beta:.4f}')
print(f'predictive var at x=0 (in)  : {v_in:.4f}')
print(f'predictive var at x=3 (out) : {v_out:.4f}')
assert v_in >= 1.0 / beta and v_out >= 1.0 / beta, 'variance can never beat the noise floor'
assert v_out > v_in, 'epistemic uncertainty grows in extrapolation'
print('\n✅ predictive variance = irreducible noise + epistemic term that flares out-of-distribution')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **prior strength $\alpha$** | too strong over-shrinks toward the prior; too weak ignores it (→ OLS overfitting) |
| **noise precision $\beta$** | mis-set $\beta$ mis-scales the error bars; often estimated from data (empirical Bayes) |
| **conjugacy is a luxury** | the closed form needs Gaussian prior + Gaussian noise; other likelihoods need MCMC/VI |
| **linear-in-features only** | uncertainty is honest only within the model class; a wrong basis gives confidently-wrong bars |
| **extrapolation** | error bars widen, but the *mean* is still a linear extrapolation — trust it less far out |

Demo: more data drives the epistemic variance to zero but never below the noise
floor.

In [ ]:
# More data crushes the EPISTEMIC uncertainty but never the noise floor: add points and
# watch the posterior variance at a test location fall toward 1/beta and stop.
def var_after_n(n_extra, x_star=1.0):
    xg = np.linspace(-2, 2, n_extra)
    yg = 0.9 * xg + 0.3 + rng.normal(0, 0.15, size=n_extra)
    Pg = np.c_[np.ones_like(xg), xg]
    _, Sg = posterior(Pg, yg, alpha, beta)
    phi = np.r_[1.0, x_star]
    return 1.0 / beta + phi @ Sg @ phi
for n in [5, 20, 100, 1000]:
    print(f'n={n:>4}: predictive var at x=1 = {var_after_n(n):.4f}  (noise floor {1/beta:.4f})')
print('\nEpistemic uncertainty -> 0 with data; the 1/beta noise floor is irreducible.')

## ✏️ Your turn

**Exercise.** Implement `predictive(phi_star, m_N, S_N, beta)` returning the `(mean, variance)` of the
predictive distribution at a single input feature vector `phi_star`:
mean = m_Nᵀφ*, variance = β⁻¹ + φ*ᵀ S_N φ*.

In [ ]:
def predictive(phi_star, m_N, S_N, beta):
    # TODO(you): return (predictive mean, predictive variance) at phi_star
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
m0, v0 = predictive(np.array([1.0, 0.0]), m_N, S_N, beta)   # x=0, inside the data
m3, v3 = predictive(np.array([1.0, 3.0]), m_N, S_N, beta)   # x=3, far away
assert np.isclose(m0, m_N[0])                                # mean at x=0 is the bias
assert v3 > v0                                               # uncertainty grows away from data
assert v0 > 1.0 / beta                                       # always at least the noise floor
print(f'\u2713 predictive correct: var(x=0)={v0:.3f} < var(x=3)={v3:.3f}')

<details>
<summary>Solution</summary>

```python
def predictive(phi_star, m_N, S_N, beta):
    mean = phi_star @ m_N
    var = 1.0 / beta + phi_star @ S_N @ phi_star
    return mean, var
```

The variance is the noise floor β⁻¹ plus a model-uncertainty term φ*ᵀS_Nφ* that grows as φ* moves
into regions the posterior is unsure about — which is exactly where there's little data.

</details>

## Key takeaways

- **Bayesian regression = a distribution over lines**, not one line — you get
  calibrated error bars for free.
- **The posterior mean IS ridge** ($\lambda=\alpha/\beta$); as the prior vanishes
  it becomes **OLS** (we verified both).
- **Predictive variance = noise floor $1/\beta$ + epistemic term.** The epistemic
  part shrinks with data and flares in extrapolation; the noise floor is
  irreducible.
- **This is why Bayesian models "know what they don't know"** — the widening bars
  away from data are the honest uncertainty a point estimate hides.